# Tutorial: Creating a Hybrid Retrieval Pipeline

- **Level**: Intermediate
- **Time to complete**: 15 minutes
- **Components Used**: [`DocumentSplitter`](https://docs.haystack.deepset.ai/docs/documentsplitter), [`SentenceTransformersDocumentEmbedder`](https://docs.haystack.deepset.ai/docs/sentencetransformersdocumentembedder), [`InMemoryDocumentStore`](https://docs.haystack.deepset.ai/docs/inmemorydocumentstore), [`InMemoryBM25Retriever`](https://docs.haystack.deepset.ai/docs/inmemorybm25retriever), [`InMemoryEmbeddingRetriever`](https://docs.haystack.deepset.ai/docs/inmemoryembeddingretriever), and [`SentenceTransformersSimilarityRanker`](https://docs.haystack.deepset.ai/docs/sentencetransformerssimilarityranker)
- **Prerequisites**: None
- **Goal**: After completing this tutorial, you will have learned about creating a hybrid retrieval and when it's useful.

## Overview

**Hybrid Retrieval** combines keyword-based and embedding-based retrieval techniques, leveraging the strengths of both approaches. In essence, dense embeddings excel in grasping the contextual nuances of the query, while keyword-based methods excel in matching keywords.

There are many cases when a simple keyword-based approaches like BM25 performs better than a dense retrieval (for example in a specific domain like healthcare) because a dense model needs to be trained on data. For more details about Hybrid Retrieval, check out [Blog Post: Hybrid Document Retrieval](https://haystack.deepset.ai/blog/hybrid-retrieval).

## Installing Haystack

Install Haystack and other required packages with `pip`:

In [1]:
%pip install haystack-ai==3.5.0 "datasets>=2.6.1" sentence-transformers-haystack


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 785.7/785.7 kB 30.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 92.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 70.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 23.4 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [sentence-transformers-haystack]


In [2]:
import sys
import platform
import subprocess
from pathlib import Path
import importlib.metadata as metadata

print("Python:", sys.version)
print("Platform:", platform.platform())
for package in ("haystack-ai", "sentence-transformers-haystack", "datasets", "sentence-transformers", "transformers", "torch", "numpy", "requests"):
    print(f"{package}=={metadata.version(package)}")
Path("requirements-lock.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    encoding="utf-8",
)


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
haystack-ai==3.5.0
sentence-transformers-haystack==2.2.0
datasets==4.8.5
sentence-transformers==6.2.0
transformers==5.22.1
torch==2.14.1+cu130
numpy==2.4.6
requests==2.32.4


14303

## Initializing the DocumentStore

You'll start creating your question answering system by initializing a DocumentStore. A DocumentStore stores the Documents that your system uses to find answers to your questions. In this tutorial, you'll be using the [`InMemoryDocumentStore`](https://docs.haystack.deepset.ai/docs/inmemorydocumentstore).

In [3]:
import torch
from haystack.utils import ComponentDevice
from haystack.document_stores.in_memory import InMemoryDocumentStore

DEVICE = ComponentDevice.from_str("cuda:0" if torch.cuda.is_available() else "cpu")
document_store = InMemoryDocumentStore()


> `InMemoryDocumentStore` is the simplest DocumentStore to get started with. It requires no external dependencies and it's a good option for smaller projects and debugging. But it doesn't scale up so well to larger Document collections, so it's not a good choice for production systems. To learn more about the different types of external databases that Haystack supports, see [DocumentStore Integrations](https://haystack.deepset.ai/integrations?type=Document+Store&version=2.0).

## Fetching and Processing Documents

As Documents, you will use the PubMed Abstracts. There are a lot of datasets from PubMed on Hugging Face Hub; you will use [anakin87/medrag-pubmed-chunk](https://huggingface.co/datasets/anakin87/medrag-pubmed-chunk) in this tutorial.

Then, you will create Documents from the dataset with a simple for loop.
Each data point in the PubMed dataset has 4 features:
* *pmid*
* *title*
* *content*: the abstract
* *contents*: abstract + title

For searching, you will use the *contents* feature. The other features will be stored as metadata, and you will use them to have a **pretty print** of the search results or for [metadata filtering](https://docs.haystack.deepset.ai/docs/metadata-filtering).

In [4]:
from datasets import load_dataset
from haystack import Document

dataset = load_dataset("anakin87/medrag-pubmed-chunk", split="train[:100]")

docs = []
for doc in dataset:
    docs.append(
        Document(content=doc["contents"], meta={"title": doc["title"], "abstract": doc["content"], "pmid": doc["id"]})
    )

README.md:   0%|          | 0.00/383 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.4MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15377 [00:00<?, ? examples/s]

## Indexing Documents with a Pipeline

Create a pipeline to store the data in the document store with their embedding. For this pipeline, you need a [DocumentSplitter](https://docs.haystack.deepset.ai/docs/documentsplitter) to split documents into chunks of 512 words, [SentenceTransformersDocumentEmbedder](https://docs.haystack.deepset.ai/docs/sentencetransformersdocumentembedder) to create document embeddings for dense retrieval and [DocumentWriter](https://docs.haystack.deepset.ai/docs/documentwriter) to write documents to the document store.

As an embedding model, you will use [BAAI/bge-small-en-v1.5](https://huggingface.co/BAAI/bge-small-en-v1.5) on Hugging Face. Feel free to test other models on Hugging Face or use another [Embedder](https://docs.haystack.deepset.ai/docs/embedders) to switch the model provider.

> If this step takes too long for you, replace the embedding model with a smaller model such as `sentence-transformers/all-MiniLM-L6-v2` or `sentence-transformers/all-mpnet-base-v2`. Make sure that the `split_length` is updated according to your model's token limit.

In [5]:
from haystack.components.writers import DocumentWriter
from haystack_integrations.components.embedders.sentence_transformers import SentenceTransformersDocumentEmbedder
from haystack.components.preprocessors.document_splitter import DocumentSplitter
from haystack import Pipeline
from haystack.utils import ComponentDevice

document_splitter = DocumentSplitter(split_by="word", split_length=512, split_overlap=32)
document_embedder = SentenceTransformersDocumentEmbedder(
    model="BAAI/bge-small-en-v1.5", device=DEVICE
)
document_writer = DocumentWriter(document_store)

indexing_pipeline = Pipeline()
indexing_pipeline.add_component("document_splitter", document_splitter)
indexing_pipeline.add_component("document_embedder", document_embedder)
indexing_pipeline.add_component("document_writer", document_writer)

indexing_pipeline.connect("document_splitter", "document_embedder")
indexing_pipeline.connect("document_embedder", "document_writer")

indexing_pipeline.run({"document_splitter": {"documents": docs}})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

{'document_writer': {'documents_written': 100}}

Documents are stored in `InMemoryDocumentStore` with their embeddings, now it's time for creating the hybrid retrieval pipeline ✅

## Creating a Pipeline for Hybrid Retrieval

Hybrid retrieval refers to the combination of multiple retrieval methods to enhance overall performance. In the context of search systems, a hybrid retrieval pipeline executes both traditional keyword-based search and dense vector search, later ranking the results with a cross-encoder model. This combination allows the search system to leverage the strengths of different approaches, providing more accurate and diverse results.

Here are the required steps for a hybrid retrieval pipeline:

### 1) Initialize Retrievers and the Embedder

Initialize a [InMemoryEmbeddingRetriever](https://docs.haystack.deepset.ai/docs/inmemoryembeddingretriever) and [InMemoryBM25Retriever](https://docs.haystack.deepset.ai/docs/inmemorybm25retriever) to perform both dense and keyword-based retrieval. For dense retrieval, you also need a [SentenceTransformersTextEmbedder](https://docs.haystack.deepset.ai/docs/sentencetransformerstextembedder) that computes the embedding of the search query by using the same embedding model `BAAI/bge-small-en-v1.5` that was used in the indexing pipeline:

In [6]:
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever, InMemoryEmbeddingRetriever
from haystack_integrations.components.embedders.sentence_transformers import SentenceTransformersTextEmbedder

text_embedder = SentenceTransformersTextEmbedder(
    model="BAAI/bge-small-en-v1.5", device=DEVICE
)
embedding_retriever = InMemoryEmbeddingRetriever(document_store)
bm25_retriever = InMemoryBM25Retriever(document_store)

### 2) Rank the Results

Use the [SentenceTransformersSimilarityRanker](https://docs.haystack.deepset.ai/docs/sentencetransformerssimilarityranker) that scores the relevancy of all retrieved documents for the given search query by using a cross encoder model. In this example, you will use [BAAI/bge-reranker-base](https://huggingface.co/BAAI/bge-reranker-base) model to rank the retrieved documents but you can replace this model with other cross-encoder models on Hugging Face.

In [7]:
from haystack_integrations.components.rankers.sentence_transformers import SentenceTransformersSimilarityRanker

ranker = SentenceTransformersSimilarityRanker(model="BAAI/bge-reranker-base")

### 3) Create the Hybrid Retrieval Pipeline

Add all initialized components to your pipeline. Both the BM25 and Embedding retrievers are connected to the same ranker input.

In [8]:
from haystack import Pipeline

hybrid_retrieval = Pipeline()
hybrid_retrieval.add_component("text_embedder", text_embedder)
hybrid_retrieval.add_component("embedding_retriever", embedding_retriever)
hybrid_retrieval.add_component("bm25_retriever", bm25_retriever)
hybrid_retrieval.add_component("ranker", ranker)

hybrid_retrieval.connect("text_embedder", "embedding_retriever")
hybrid_retrieval.connect("bm25_retriever", "ranker")
hybrid_retrieval.connect("embedding_retriever", "ranker")

🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - embedding_retriever: InMemoryEmbeddingRetriever
  - bm25_retriever: InMemoryBM25Retriever
  - ranker: SentenceTransformersSimilarityRanker
🛤️ Connections
  - text_embedder.embedding -> embedding_retriever.query_embedding (list[float])
  - embedding_retriever.documents -> ranker.documents (list[Document])
  - bm25_retriever.documents -> ranker.documents (list[Document])

### 5) Visualize the Pipeline (Optional)

To understand how you formed a hybrid retrieval pipeline, use [draw()](https://docs.haystack.deepset.ai/docs/visualizing-pipelines) method of the pipeline. If you're running this notebook on Google Colab, the generate file will be saved in "Files" section on the sidebar.

In [9]:
# hybrid_retrieval.draw("hybrid-retrieval.png")

## Testing the Hybrid Retrieval

Pass the query to `text_embedder`, `bm25_retriever` and `ranker` and run the retrieval pipeline:


In [10]:
query = "hemoglobin oxygen affinity"

result = hybrid_retrieval.run(
    {
        "text_embedder": {"text": query},
        "bm25_retriever": {"query": query},
        "ranker": {"query": query}
    }
)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
for i, doc in enumerate(result["ranker"]["documents"][:5], 1):
    print(f"排名 {i} | 分数：{doc.score:.4f}")
    print(f"内容：{doc.content[:200]}")
    print("-" * 50)

排名 1 | 分数：0.9814
内容：The oxygen affinity of haemoglobin Tak, a variant with an elongated beta chain. The oxygen affinity was investigated of purified Hb Tak, a human haemoglobin variant with elongated beta-chains. A very 
--------------------------------------------------
排名 2 | 分数：0.9387
内容：Studies on electron transfer between mercury electrode and hemoprotein. The electrochemical behaviour of ferricytochrome c, metmyoglobin and methemoglobin was studied using d.c., a.c. and differential
--------------------------------------------------
排名 3 | 分数：0.9294
内容：Haemoglobin Rahere (beta Lys-Thr): A new high affinity haemoglobin associated with decreased 2, 3-diphosphoglycerate binding and relative polycythaemia. A new haemoglobin with increased oxygen affinit
--------------------------------------------------
排名 4 | 分数：0.0304
内容：The interaction of bovine erythrocyte superoxide dismutase with hydrogen peroxide: inactivation of the enzyme. Bovine erythrocyte superoxide dismutase was slowly an

In [12]:
query = "hemoglobin oxygen affinity"

bm25_result = bm25_retriever.run(
    query=query,
    top_k=5
)

for i, doc in enumerate(bm25_result["documents"], 1):
    print(f"排名 {i} | BM25 分数：{doc.score:.4f}")
    print(f"内容：{doc.content[:200]}")
    print("-" * 50)

排名 1 | BM25 分数：10.8053
内容：Studies on electron transfer between mercury electrode and hemoprotein. The electrochemical behaviour of ferricytochrome c, metmyoglobin and methemoglobin was studied using d.c., a.c. and differential
--------------------------------------------------
排名 2 | BM25 分数：9.8993
内容：Haemoglobin Rahere (beta Lys-Thr): A new high affinity haemoglobin associated with decreased 2, 3-diphosphoglycerate binding and relative polycythaemia. A new haemoglobin with increased oxygen affinit
--------------------------------------------------
排名 3 | BM25 分数：9.4093
内容：The oxygen affinity of haemoglobin Tak, a variant with an elongated beta chain. The oxygen affinity was investigated of purified Hb Tak, a human haemoglobin variant with elongated beta-chains. A very 
--------------------------------------------------
排名 4 | BM25 分数：5.9933
内容：The interaction of bovine erythrocyte superoxide dismutase with hydrogen peroxide: inactivation of the enzyme. Bovine erythrocyte superoxide di

In [13]:
query = "hemoglobin oxygen affinity"

# 把查询转换成向量
embedding = text_embedder.run(text=query)["embedding"]

# 只使用 Embedding Retriever 检索
embedding_result = embedding_retriever.run(
    query_embedding=embedding,
    top_k=5
)

for i, doc in enumerate(embedding_result["documents"], 1):
    print(f"排名 {i} | Embedding 分数：{doc.score:.4f}")
    print(f"内容：{doc.content[:200]}")
    print("-" * 50)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

排名 1 | Embedding 分数：0.8304
内容：The oxygen affinity of haemoglobin Tak, a variant with an elongated beta chain. The oxygen affinity was investigated of purified Hb Tak, a human haemoglobin variant with elongated beta-chains. A very 
--------------------------------------------------
排名 2 | Embedding 分数：0.7967
内容：Haemoglobin Rahere (beta Lys-Thr): A new high affinity haemoglobin associated with decreased 2, 3-diphosphoglycerate binding and relative polycythaemia. A new haemoglobin with increased oxygen affinit
--------------------------------------------------
排名 3 | Embedding 分数：0.7606
内容：Studies on electron transfer between mercury electrode and hemoprotein. The electrochemical behaviour of ferricytochrome c, metmyoglobin and methemoglobin was studied using d.c., a.c. and differential
--------------------------------------------------
排名 4 | Embedding 分数：0.7443
内容：Kinetic light scattering studies on the dissociation of hemoglobin from Lumbricus terrestris. The kinetics of the pH-induced

In [14]:
import requests
import zipfile
import io
import json
import csv

url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"

response = requests.get(url, timeout=120)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    corpus = [
        json.loads(line)
        for line in z.read("scifact/corpus.jsonl").decode("utf-8").splitlines()
    ]

    queries = [
        json.loads(line)
        for line in z.read("scifact/queries.jsonl").decode("utf-8").splitlines()
    ]

    qrels = list(
        csv.DictReader(
            io.StringIO(z.read("scifact/qrels/test.tsv").decode("utf-8")),
            delimiter="\t"
        )
    )

print("文档数量：", len(corpus))
print("查询数量：", len(queries))
print("测试集相关性标注数量：", len(qrels))

文档数量： 5183
查询数量： 1109
测试集相关性标注数量： 339


In [15]:
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore

scifact_store = InMemoryDocumentStore()

scifact_docs = [
    Document(
        id=str(doc["_id"]),
        content=doc["title"] + "\n" + doc["text"],
        meta={"title": doc["title"]}
    )
    for doc in corpus
]

print("SciFact 文档数量：", len(scifact_docs))
print("文档 ID 示例：", scifact_docs[0].id)

SciFact 文档数量： 5183
文档 ID 示例： 4983


In [16]:
print("SciFact 文档库数量：", scifact_store.count_documents())

SciFact 文档库数量： 0


In [17]:
from haystack import Pipeline
from haystack.components.writers import DocumentWriter
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder
)
from haystack.utils import ComponentDevice

scifact_embedder = SentenceTransformersDocumentEmbedder(
    model="BAAI/bge-small-en-v1.5",
    device=DEVICE,
    batch_size=16
)

scifact_writer = DocumentWriter(scifact_store)

scifact_indexing = Pipeline()
scifact_indexing.add_component("embedder", scifact_embedder)
scifact_indexing.add_component("writer", scifact_writer)

scifact_indexing.connect("embedder.documents", "writer.documents")

index_result = scifact_indexing.run({
    "embedder": {"documents": scifact_docs}
})

print("索引结果：", index_result)
print("文档库数量：", scifact_store.count_documents())

Batches:   0%|          | 0/324 [00:00<?, ?it/s]

索引结果： {'writer': {'documents_written': 5183}}
文档库数量： 5183


In [18]:
from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever,
    InMemoryEmbeddingRetriever
)

from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersTextEmbedder
)

# 查询向量化模型
scifact_text_embedder = SentenceTransformersTextEmbedder(
    model="BAAI/bge-small-en-v1.5",
    device=DEVICE
)

# BM25 关键词检索器
scifact_bm25_retriever = InMemoryBM25Retriever(
    document_store=scifact_store,
    top_k=10
)

# Embedding 语义检索器
scifact_embedding_retriever = InMemoryEmbeddingRetriever(
    document_store=scifact_store,
    top_k=10
)

print("SciFact 检索组件创建成功")
scifact_text_embedder.warm_up()


SciFact 检索组件创建成功


In [19]:
# 建立查询 ID 到查询文本的映射
query_map = {str(q["_id"]): q["text"] for q in queries}

# 从测试集 qrels 中选择一条有标注的查询
test_query_id = qrels[0]["query-id"]
test_query = query_map[test_query_id]

print("查询 ID：", test_query_id)
print("查询内容：", test_query)

# BM25 检索
bm25_result = scifact_bm25_retriever.run(
    query=test_query,
    top_k=10
)

# Embedding 检索
query_embedding = scifact_text_embedder.run(
    text=test_query
)["embedding"]

embedding_result = scifact_embedding_retriever.run(
    query_embedding=query_embedding,
    top_k=10
)

# 查看两种方法的 Top 3 文档 ID
print("\nBM25 Top 3：")
for doc in bm25_result["documents"][:3]:
    print(doc.id, doc.score)

print("\nEmbedding Top 3：")
for doc in embedding_result["documents"][:3]:
    print(doc.id, doc.score)

查询 ID： 1
查询内容： 0-dimensional biomaterials show inductive properties.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


BM25 Top 3：
43385013 12.177518155822892
10931595 11.63081251875392
10608397 10.943844407076373

Embedding Top 3：
4346436 0.7819477962198187
17388232 0.7781149705869497
29638116 0.736392214189239


In [20]:
# 获取当前查询的标准相关文档 ID
relevant_ids = {
    row["corpus-id"]
    for row in qrels
    if row["query-id"] == test_query_id
    and int(row["score"]) > 0
}

# 获取两种方法的 Top 10 文档 ID
bm25_ids = [
    doc.id for doc in bm25_result["documents"]
]

embedding_ids = [
    doc.id for doc in embedding_result["documents"]
]

print("标准相关文档：", relevant_ids)

print("BM25 Top 10 命中：",
      relevant_ids.intersection(bm25_ids))

print("Embedding Top 10 命中：",
      relevant_ids.intersection(embedding_ids))

标准相关文档： {'31715818'}
BM25 Top 10 命中： set()
Embedding Top 10 命中： {'31715818'}


In [21]:
import numpy as np

# 1. 获取测试集全部查询 ID
test_query_ids = sorted({
    row["query-id"] for row in qrels
})

# 2. 建立标准答案映射
qrels_map = {}

for row in qrels:
    qid = row["query-id"]
    did = row["corpus-id"]

    if int(row["score"]) > 0:
        qrels_map.setdefault(qid, set()).add(did)

# 3. 建立查询文本映射
query_map = {
    str(q["_id"]): q["text"]
    for q in queries
}

corpus_ids = {doc.id for doc in scifact_docs}
assert len(corpus_ids) == len(scifact_docs), "Duplicate corpus IDs"
assert scifact_store.count_documents() == len(scifact_docs)
assert set(test_query_ids) <= set(query_map), "Missing query text"
assert all(qrels_map.get(qid) for qid in test_query_ids), "Missing positive qrels"
assert all(qrels_map[qid] <= corpus_ids for qid in test_query_ids), "Unknown relevant document"

# 4. 计算 Recall@10 和 MRR@10
def calculate_metrics(retriever_name):
    recalls = []
    mrrs = []

    for qid in test_query_ids:
        query = query_map[qid]
        relevant = qrels_map[qid]

        if retriever_name == "BM25":
            docs = scifact_bm25_retriever.run(
                query=query,
                top_k=10
            )["documents"]

        elif retriever_name == "Embedding":
            embedding = scifact_text_embedder.run(
                text=query
            )["embedding"]

            docs = scifact_embedding_retriever.run(
                query_embedding=embedding,
                top_k=10
            )["documents"]

        else:
            raise ValueError(f"Unknown retriever: {retriever_name}")

        retrieved_ids = [doc.id for doc in docs[:10]]

        hits = len(set(retrieved_ids) & relevant)
        recalls.append(hits / len(relevant))

        reciprocal_rank = 0

        for rank, doc_id in enumerate(retrieved_ids, 1):
            if doc_id in relevant:
                reciprocal_rank = 1 / rank
                break

        mrrs.append(reciprocal_rank)

    return {
        "Recall@10": np.mean(recalls),
        "MRR@10": np.mean(mrrs)
    }

# 5. 分别评测
bm25_metrics = calculate_metrics("BM25")
embedding_metrics = calculate_metrics("Embedding")

print("测试查询数量：", len(test_query_ids))
print("BM25：", bm25_metrics)
print("Embedding：", embedding_metrics)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (291 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

测试查询数量： 300
BM25： {'Recall@10': np.float64(0.7851111111111111), 'MRR@10': np.float64(0.6225277777777778)}
Embedding： {'Recall@10': np.float64(0.8452222222222222), 'MRR@10': np.float64(0.6844708994708995)}


In [22]:

# 四类查询的 ID 集合
both_hit_ids = []
bm25_only_ids = []
embedding_only_ids = []
neither_hit_ids = []

for qid in test_query_ids:

    query = query_map[qid]
    relevant = qrels_map[qid]

    # BM25 检索
    bm25_docs = scifact_bm25_retriever.run(
        query=query,
        top_k=10
    )["documents"]

    # Embedding 检索
    query_embedding = scifact_text_embedder.run(
        text=query
    )["embedding"]

    embedding_docs = scifact_embedding_retriever.run(
        query_embedding=query_embedding,
        top_k=10
    )["documents"]

    # 判断是否命中标准相关文档
    bm25_hit = any(
        doc.id in relevant for doc in bm25_docs
    )

    embedding_hit = any(
        doc.id in relevant for doc in embedding_docs
    )

    # 根据命中情况分类
    if bm25_hit and embedding_hit:
        both_hit_ids.append(qid)

    elif bm25_hit:
        bm25_only_ids.append(qid)

    elif embedding_hit:
        embedding_only_ids.append(qid)

    else:
        neither_hit_ids.append(qid)


print("两种方法都命中：", len(both_hit_ids))
print("仅 BM25 命中：", len(bm25_only_ids))
print("仅 Embedding 命中：", len(embedding_only_ids))
print("两种方法都未命中：", len(neither_hit_ids))

print("\n仅 BM25 命中的查询 ID：", bm25_only_ids[:5])
print("仅 Embedding 命中的查询 ID：", embedding_only_ids[:5])
print("两种方法都未命中的查询 ID：", neither_hit_ids[:5])


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (291 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

两种方法都命中： 225
仅 BM25 命中： 17
仅 Embedding 命中： 32
两种方法都未命中： 26

仅 BM25 命中的查询 ID： ['1100', '1179', '128', '1362', '1363']
仅 Embedding 命中的查询 ID： ['1', '1049', '1088', '1140', '1196']
两种方法都未命中的查询 ID： ['1110', '1175', '1191', '1199', '1213']


In [23]:
from haystack import Pipeline
from haystack_integrations.components.rankers.sentence_transformers import (
    SentenceTransformersSimilarityRanker
)
from haystack.utils import ComponentDevice

# 创建重排序模型
scifact_ranker = SentenceTransformersSimilarityRanker(
    model="BAAI/bge-reranker-base",
    device=DEVICE,
    top_k=10
)

# 创建混合检索流水线
scifact_hybrid = Pipeline()

scifact_hybrid.add_component(
    "text_embedder", scifact_text_embedder
)

scifact_hybrid.add_component(
    "bm25_retriever", scifact_bm25_retriever
)

scifact_hybrid.add_component(
    "embedding_retriever", scifact_embedding_retriever
)

scifact_hybrid.add_component(
    "ranker", scifact_ranker
)

# 连接查询向量化与向量检索
scifact_hybrid.connect(
    "text_embedder.embedding",
    "embedding_retriever.query_embedding"
)

# 将两路检索结果送入 Ranker
scifact_hybrid.connect(
    "bm25_retriever.documents",
    "ranker.documents"
)

scifact_hybrid.connect(
    "embedding_retriever.documents",
    "ranker.documents"
)

print("SciFact Hybrid Pipeline 创建成功")

SciFact Hybrid Pipeline 创建成功


In [24]:
# 使用之前验证过的 SciFact 测试查询
query_id = "1"
query_text = query_map[query_id]

# 执行混合检索
hybrid_result = scifact_hybrid.run({
    "text_embedder": {"text": query_text},
    "bm25_retriever": {"query": query_text},
    "ranker": {"query": query_text}
})

# 查看最终 Top 10 文档
hybrid_docs = hybrid_result["ranker"]["documents"]

print("查询：", query_text)
print("标准相关文档：", qrels_map[query_id])

print("\nHybrid + Ranker Top 10：")
for i, doc in enumerate(hybrid_docs[:10], 1):
    print(f"排名 {i} | ID: {doc.id} | 分数: {doc.score:.4f}")

# 检查是否命中
retrieved_ids = {doc.id for doc in hybrid_docs[:10]}

print("\n命中的相关文档：",
      retrieved_ids & qrels_map[query_id])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

查询： 0-dimensional biomaterials show inductive properties.
标准相关文档： {'31715818'}

Hybrid + Ranker Top 10：
排名 1 | ID: 4346436 | 分数: 0.0042
排名 2 | ID: 10931595 | 分数: 0.0014
排名 3 | ID: 13231899 | 分数: 0.0012
排名 4 | ID: 21456232 | 分数: 0.0011
排名 5 | ID: 803312 | 分数: 0.0005
排名 6 | ID: 40212412 | 分数: 0.0005
排名 7 | ID: 7583104 | 分数: 0.0004
排名 8 | ID: 10608397 | 分数: 0.0003
排名 9 | ID: 43385013 | 分数: 0.0003
排名 10 | ID: 29638116 | 分数: 0.0003

命中的相关文档： set()


In [25]:
query_text = query_map["1"]
hybrid_docs = scifact_hybrid.run({
    "text_embedder": {"text": query_text},
    "bm25_retriever": {"query": query_text},
    "ranker": {"query": query_text, "top_k": 10}
})["ranker"]["documents"]
target_id = "31715818"

bm25_docs = scifact_bm25_retriever.run(
    query=query_text, top_k=10
)["documents"]

query_vector = scifact_text_embedder.run(
    text=query_text
)["embedding"]

embedding_docs = scifact_embedding_retriever.run(
    query_embedding=query_vector, top_k=10
)["documents"]

def find_rank(documents, target_id):
    for rank, doc in enumerate(documents, 1):
        if doc.id == target_id:
            return rank
    return None

print("标准相关文档 ID：", target_id)
print("BM25 中的排名：", find_rank(bm25_docs, target_id))
print("Embedding 中的排名：", find_rank(embedding_docs, target_id))
print("Hybrid + Ranker 中的排名：", find_rank(hybrid_docs, target_id))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

标准相关文档 ID： 31715818
BM25 中的排名： None
Embedding 中的排名： 6
Hybrid + Ranker 中的排名： None


In [26]:
query_text = query_map["1"]

# 重新运行当前查询
hybrid_full_result = scifact_hybrid.run({
    "text_embedder": {"text": query_text},
    "bm25_retriever": {"query": query_text},
    "ranker": {"query": query_text, "top_k": 20}
})

full_docs = hybrid_full_result["ranker"]["documents"]

# 查找标准相关文档的最终排名
target_rank = None

for i, doc in enumerate(full_docs, 1):
    if doc.id == "31715818":
        target_rank = i
        print(f"正确文档排名：{i}")
        print(f"Ranker 分数：{doc.score:.6f}")
        break

if target_rank is None:
    print("正确文档未出现在 Ranker 返回的候选结果中")

print("Ranker 返回文档数量：", len(full_docs))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

正确文档排名：15
Ranker 分数：0.000093
Ranker 返回文档数量： 19


In [27]:
import numpy as np

hybrid_recalls = []
hybrid_mrrs = []

for qid in test_query_ids:
    query = query_map[qid]
    relevant = qrels_map[qid]

    # 执行混合检索与重排序
    result = scifact_hybrid.run({
        "text_embedder": {"text": query},
        "bm25_retriever": {"query": query},
        "ranker": {"query": query, "top_k": 10}
    })

    # 获取最终 Top 10
    retrieved_ids = [
        doc.id
        for doc in result["ranker"]["documents"][:10]
    ]

    # 计算 Recall@10
    hits = len(set(retrieved_ids) & relevant)
    hybrid_recalls.append(hits / len(relevant))

    # 计算 MRR@10
    reciprocal_rank = 0

    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant:
            reciprocal_rank = 1 / rank
            break

    hybrid_mrrs.append(reciprocal_rank)

print("测试查询数量：", len(hybrid_recalls))
print("Hybrid Recall@10：", np.mean(hybrid_recalls))
print("Hybrid MRR@10：", np.mean(hybrid_mrrs))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (291 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

测试查询数量： 300
Hybrid Recall@10： 0.8782222222222223
Hybrid MRR@10： 0.699026455026455


In [28]:
def rrf_retrieval(query, top_k=10, rrf_k=60):

    bm25_docs = scifact_bm25_retriever.run(
        query=query, top_k=10
    )["documents"]

    query_embedding = scifact_text_embedder.run(
        text=query
    )["embedding"]

    embedding_docs = scifact_embedding_retriever.run(
        query_embedding=query_embedding,
        top_k=10
    )["documents"]

    doc_map = {}
    rrf_scores = {}

    for docs in [bm25_docs, embedding_docs]:
        for rank, doc in enumerate(docs, start=1):
            doc_id = str(doc.id)
            doc_map[doc_id] = doc

            rrf_scores[doc_id] = (
                rrf_scores.get(doc_id, 0)
                + 1 / (rrf_k + rank)
            )

    ranked_ids = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )

    return [
        (doc_id, rrf_scores[doc_id], doc_map[doc_id])
        for doc_id in ranked_ids[:top_k]
    ]

print("RRF 检索函数创建成功")

RRF 检索函数创建成功


In [29]:
# 使用之前的 SciFact 测试查询
query_id = "1"
query_text = query_map[query_id]

# 执行 RRF
rrf_results = rrf_retrieval(query_text)

print("查询：", query_text)

print("\nRRF Top 10：")

for rank, (doc_id, score, doc) in enumerate(rrf_results, 1):
    print(
        f"排名 {rank} | ID: {doc_id} | "
        f"RRF 分数: {score:.6f}"
    )

# 检查是否命中标准答案
relevant_ids = qrels_map[query_id]

rrf_ids = {
    doc_id for doc_id, score, doc in rrf_results
}

print("\n标准相关文档：", relevant_ids)
print("RRF 命中的相关文档：", rrf_ids & relevant_ids)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

查询： 0-dimensional biomaterials show inductive properties.

RRF Top 10：
排名 1 | ID: 40212412 | RRF 分数: 0.030118
排名 2 | ID: 43385013 | RRF 分数: 0.016393
排名 3 | ID: 4346436 | RRF 分数: 0.016393
排名 4 | ID: 10931595 | RRF 分数: 0.016129
排名 5 | ID: 17388232 | RRF 分数: 0.016129
排名 6 | ID: 10608397 | RRF 分数: 0.015873
排名 7 | ID: 29638116 | RRF 分数: 0.015873
排名 8 | ID: 21456232 | RRF 分数: 0.015625
排名 9 | ID: 27049238 | RRF 分数: 0.015385
排名 10 | ID: 7583104 | RRF 分数: 0.015385

标准相关文档： {'31715818'}
RRF 命中的相关文档： set()


In [30]:
query_text = query_map["1"]

# 获取 RRF 完整候选排序（最多20篇）
rrf_full_results = rrf_retrieval(
    query_text,
    top_k=20
)

target_id = "31715818"

target_rank = None

for rank, (doc_id, score, doc) in enumerate(
    rrf_full_results, 1
):
    if doc_id == target_id:
        target_rank = rank
        print("正确文档在 RRF 中的排名：", rank)
        print("RRF 分数：", score)
        break

if target_rank is None:
    print("正确文档没有进入 RRF 候选集合")

print("RRF 候选文档数量：", len(rrf_full_results))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

正确文档在 RRF 中的排名： 12
RRF 分数： 0.015151515151515152
RRF 候选文档数量： 19


In [31]:
import numpy as np

rrf_recalls = []
rrf_mrrs = []

for qid in test_query_ids:

    query = query_map[qid]
    relevant = qrels_map[qid]

    # RRF 检索，最终返回 Top 10
    results = rrf_retrieval(
        query,
        top_k=10
    )

    retrieved_ids = [
        doc_id for doc_id, score, doc in results
    ]

    # Recall@10
    hits = len(set(retrieved_ids) & relevant)
    rrf_recalls.append(hits / len(relevant))

    # MRR@10
    reciprocal_rank = 0

    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant:
            reciprocal_rank = 1 / rank
            break

    rrf_mrrs.append(reciprocal_rank)

print("测试查询数量：", len(rrf_recalls))
print("RRF Recall@10：", np.mean(rrf_recalls))
print("RRF MRR@10：", np.mean(rrf_mrrs))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (291 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

测试查询数量： 300
RRF Recall@10： 0.8605555555555554
RRF MRR@10： 0.6875436507936509


In [32]:
qid = "1100"
query = query_map[qid]
relevant_ids = qrels_map[qid]

print("查询：", query)

# 根据文档 ID 查找标准相关文档
for doc_id in relevant_ids:
    documents = scifact_store.filter_documents(
        filters={
            "field": "id",
            "operator": "==",
            "value": doc_id
        }
    )

    for doc in documents:
        print("\n标准相关文档 ID：", doc.id)
        print("标题：", doc.meta.get("title"))
        print("正文：", doc.content[:1500])

查询： Statins increase blood cholesterol.

标准相关文档 ID： 7662206
标题： A Century of Cholesterol and Coronaries: From Plaques to Genes to Statins
正文： A Century of Cholesterol and Coronaries: From Plaques to Genes to Statins
One-fourth of all deaths in industrialized countries result from coronary heart disease. A century of research has revealed the essential causative agent: cholesterol-carrying low-density lipoprotein (LDL). LDL is controlled by specific receptors (LDLRs) in liver that remove it from blood. Mutations that eliminate LDLRs raise LDL and cause heart attacks in childhood, whereas mutations that raise LDLRs reduce LDL and diminish heart attacks. If we are to eliminate coronary disease, lowering LDL should be the primary goal. Effective means to achieve this goal are currently available. The key questions are: who to treat, when to treat, and how long to treat.


In [33]:
qid = "1100"
query = query_map[qid]
relevant = qrels_map[qid]

bm25_docs = scifact_bm25_retriever.run(
    query=query,
    top_k=10
)["documents"]

query_embedding = scifact_text_embedder.run(
    text=query
)["embedding"]

embedding_docs = scifact_embedding_retriever.run(
    query_embedding=query_embedding,
    top_k=10
)["documents"]

for name, documents in [
    ("BM25", bm25_docs),
    ("Embedding", embedding_docs)
]:
    print(f"\n{name} Top 10：")

    for rank, doc in enumerate(documents, 1):
        mark = "★ 标准相关文档" if doc.id in relevant else ""
        print(
            f"{rank}. ID={doc.id}, "
            f"score={doc.score:.4f} {mark}"
        )
        print("标题：", doc.meta.get("title"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


BM25 Top 10：
1. ID=21557614, score=19.4693 
标题： Pleiotropic effects of statins.
2. ID=7662206, score=18.7830 ★ 标准相关文档
标题： A Century of Cholesterol and Coronaries: From Plaques to Genes to Statins
3. ID=22420524, score=15.6237 
标题： Statin use and risk of gallstone disease followed by cholecystectomy.
4. ID=9617381, score=15.0714 
标题： Serum plant and other noncholesterol sterols, cholesterol metabolism and 22-year mortality among middle-aged men.
5. ID=30805636, score=14.9284 
标题： Peroxisome proliferator-activated receptor ligand bezafibrate for prevention of type 2 diabetes mellitus in patients with coronary artery disease.
6. ID=27449472, score=14.5238 
标题： Metabolic syndrome as a risk factor for diabetes.
7. ID=27466734, score=14.3546 
标题： Development and validation of QRISK3 risk prediction algorithms to estimate future risk of cardiovascular disease: prospective cohort study
8. ID=24005548, score=14.0536 
标题： The HMG-CoA reductase inhibitor simvastatin activates the protein kinase 

In [34]:
qid = "1100"
query = query_map[qid]
target_id = "7662206"

embedding_vector = scifact_text_embedder.run(
    text=query
)["embedding"]

docs_top100 = scifact_embedding_retriever.run(
    query_embedding=embedding_vector,
    top_k=100
)["documents"]

target_rank = next(
    (i for i, doc in enumerate(docs_top100, 1)
     if doc.id == target_id),
    None
)

print("标准相关文档在 Embedding 中的排名：", target_rank)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

标准相关文档在 Embedding 中的排名： 15


In [35]:
qid = "1049"
query = query_map[qid]
relevant = qrels_map[qid]

print("查询：", query)
print("标准相关文档 ID：", relevant)

bm25_docs = scifact_bm25_retriever.run(
    query=query, top_k=100
)["documents"]

embedding_vector = scifact_text_embedder.run(
    text=query
)["embedding"]

embedding_docs = scifact_embedding_retriever.run(
    query_embedding=embedding_vector,
    top_k=100
)["documents"]

for name, documents in [
    ("BM25", bm25_docs),
    ("Embedding", embedding_docs)
]:
    print(f"\n{name}：")
    for target_id in relevant:
        rank = next(
            (i for i, doc in enumerate(documents, 1)
             if doc.id == target_id),
            None
        )
        print("相关文档", target_id, "排名：", rank)

查询： Ribosomopathies have a low degree of cell and tissue specific pathology.
标准相关文档 ID： {'12486491'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


BM25：
相关文档 12486491 排名： None

Embedding：
相关文档 12486491 排名： 6


In [36]:
qid = "1049"
target_id = "12486491"

documents = scifact_store.filter_documents(
    filters={
        "field": "id",
        "operator": "==",
        "value": target_id
    }
)

for doc in documents:
    print("查询：", query_map[qid])
    print("\n标准相关文档标题：", doc.meta.get("title"))
    print("\n标准相关文档正文：")
    print(doc.content[:2000])

查询： Ribosomopathies have a low degree of cell and tissue specific pathology.

标准相关文档标题： Ribosome-Mediated Specificity in Hox mRNA Translation and Vertebrate Tissue Patterning

标准相关文档正文：
Ribosome-Mediated Specificity in Hox mRNA Translation and Vertebrate Tissue Patterning
Historically, the ribosome has been viewed as a complex ribozyme with constitutive rather than regulatory capacity in mRNA translation. Here we identify mutations of the Ribosomal Protein L38 (Rpl38) gene in mice exhibiting surprising tissue-specific patterning defects, including pronounced homeotic transformations of the axial skeleton. In Rpl38 mutant embryos, global protein synthesis is unchanged; however the translation of a select subset of Homeobox mRNAs is perturbed. Our data reveal that RPL38 facilitates 80S complex formation on these mRNAs as a regulatory component of the ribosome to confer transcript-specific translational control. We further show that Rpl38 expression is markedly enriched in regions of the 

In [37]:
import textwrap

case_docs = scifact_store.filter_documents(
    filters={"field": "id", "operator": "==", "value": "12486491"}
)
if not case_docs:
    raise ValueError("Document 12486491 is missing")
print(textwrap.fill(case_docs[0].content[:1500], width=90))


Ribosome-Mediated Specificity in Hox mRNA Translation and Vertebrate Tissue Patterning
Historically, the ribosome has been viewed as a complex ribozyme with constitutive rather
than regulatory capacity in mRNA translation. Here we identify mutations of the Ribosomal
Protein L38 (Rpl38) gene in mice exhibiting surprising tissue-specific patterning defects,
including pronounced homeotic transformations of the axial skeleton. In Rpl38 mutant
embryos, global protein synthesis is unchanged; however the translation of a select subset
of Homeobox mRNAs is perturbed. Our data reveal that RPL38 facilitates 80S complex
formation on these mRNAs as a regulatory component of the ribosome to confer transcript-
specific translational control. We further show that Rpl38 expression is markedly enriched
in regions of the embryo where loss-of-function phenotypes occur. Unexpectedly, a
ribosomal protein (RP) expression screen reveals dynamic regulation of individual RPs
within the vertebrate embryo. Colle

In [38]:

qid = "1110"
query = query_map[qid]
relevant = qrels_map[qid]

print("查询：", query)
print("标准相关文档 ID：", relevant)

# 获取标准相关文档
doc_map = {doc.id: doc for doc in scifact_docs}

for doc_id in relevant:
    doc = doc_map[doc_id]
    print("\n标准相关文档标题：", doc.meta["title"])
    print("标准相关文档正文：", doc.content[:1200])

# 检查两种检索算法的 Top 100
bm25_docs = scifact_bm25_retriever.run(
    query=query, top_k=100
)["documents"]

query_vector = scifact_text_embedder.run(
    text=query
)["embedding"]

embedding_docs = scifact_embedding_retriever.run(
    query_embedding=query_vector, top_k=100
)["documents"]

for name, documents in [
    ("BM25", bm25_docs),
    ("Embedding", embedding_docs)
]:
    print(f"\n{name}：")
    for doc_id in relevant:
        rank = next(
            (i for i, d in enumerate(documents, 1)
             if d.id == doc_id),
            None
        )
        print("正确文档排名：", rank)

    print("Top 3 文档标题：")
    for d in documents[:3]:
        print("-", d.meta.get("title"))


查询： Suboptimal nutrition is not predictive of chronic disease
标准相关文档 ID： {'13770184'}

标准相关文档标题： Global, regional, and national comparative risk assessment of 79 behavioural, environmental and occupational, and metabolic risks or clusters of risks, 1990–2015: a systematic analysis for the Global Burden of Disease Study 2015
标准相关文档正文： Global, regional, and national comparative risk assessment of 79 behavioural, environmental and occupational, and metabolic risks or clusters of risks, 1990–2015: a systematic analysis for the Global Burden of Disease Study 2015
BACKGROUND The Global Burden of Diseases, Injuries, and Risk Factors Study 2015 provides an up-to-date synthesis of the evidence for risk factor exposure and the attributable burden of disease. By providing national and subnational assessments spanning the past 25 years, this study can inform debates on the importance of addressing risks in context. METHODS We used the comparative risk assessment framework developed for previous it

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


BM25：
正确文档排名： None
Top 3 文档标题：
- A comparison of cancer screening practices in cancer survivors and in the general population: the Korean national health and nutrition examination survey (KNHANES) 2001–2007
- Accuracy of ICD-9-CM codes for identifying cardiovascular and stroke risk factors.
- Renal Disease and Pregnancy

Embedding：
正确文档排名： 99
Top 3 文档标题：
- Maternal and child undernutrition: consequences for adult health and human capital
- Dietary quality among men and women in 187 countries in 1990 and 2010: a systematic assessment
- Comparison of dietary assessment methods in a Southern French population: use of weighed records, estimated-diet records and a food-frequency questionnaire


In [39]:
target = next(
    doc for doc in corpus
    if str(doc["_id"]) == "13770184"
)

text = target["text"]

for word in ["nutrition", "diet", "chronic", "disease"]:
    pos = text.lower().find(word)

    print(f"\n关键词：{word}")

    if pos >= 0:
        print(text[max(0, pos-150):pos+300])
    else:
        print("文档正文中未出现该词")


关键词：nutrition
igh fasting plasma glucose (143·1 million [125·1 million to 163·5 million]), high BMI (120·1 million [83·8 million to 158·4 million]), childhood undernutrition (113·3 million [103·9 million to 123·4 million]), ambient particulate matter (103·1 million [90·8 million to 115·1 million]), high total cholesterol (88·7 million [74·6 million to 105·7 million]), household air pollution (85·6 million [66·7 million to 106·1 million]), alcohol use (85·0 mil

关键词：diet
5·7 million]), household air pollution (85·6 million [66·7 million to 106·1 million]), alcohol use (85·0 million [77·2 million to 93·0 million]), and diets high in sodium (83·0 million [49·3 million to 127·5 million]). From 1990 to 2015, attributable DALYs declined for micronutrient deficiencies, childhood undernutrition, unsafe sanitation and water, and household air pollution; reductions in risk-deleted DALY rates rather than reductions in expo

关键词：chronic
文档正文中未出现该词

关键词：disease
BACKGROUND The Global Burden of Dise

In [40]:
summary_rows = [
    ("BM25", float(bm25_metrics["Recall@10"]), float(bm25_metrics["MRR@10"])),
    ("Embedding", float(embedding_metrics["Recall@10"]), float(embedding_metrics["MRR@10"])),
    ("Hybrid + RRF", float(np.mean(rrf_recalls)), float(np.mean(rrf_mrrs))),
    ("Hybrid + Cross-Encoder", float(np.mean(hybrid_recalls)), float(np.mean(hybrid_mrrs))),
]
print(f"{'Method':<28} {'Recall@10':>12} {'MRR@10':>12}")
for name, recall, mrr in summary_rows:
    print(f"{name:<28} {recall:>12.4f} {mrr:>12.4f}")

import json
from pathlib import Path

metrics_output = {
    "dataset": "BEIR SciFact",
    "documents": len(scifact_docs),
    "queries": len(test_query_ids),
    "candidate_top_k_per_retriever": 10,
    "output_top_k": 10,
    "rrf_k": 60,
    "metrics": {
        name: {"Recall@10": recall, "MRR@10": mrr}
        for name, recall, mrr in summary_rows
    },
}
Path("metrics.json").write_text(
    json.dumps(metrics_output, ensure_ascii=False, indent=2), encoding="utf-8"
)


Method                          Recall@10       MRR@10
BM25                               0.7851       0.6225
Embedding                          0.8452       0.6845
Hybrid + RRF                       0.8606       0.6875
Hybrid + Cross-Encoder             0.8782       0.6990


575

## What's next

🎉 Congratulations! You've create a hybrid retrieval pipeline!

If you'd like to use this retrieval method in a RAG pipeline, check out [Tutorial: Creating Your First QA Pipeline with Retrieval-Augmentation](https://haystack.deepset.ai/tutorials/27_first_rag_pipeline) to learn about the next steps.

To stay up to date on the latest Haystack developments, you can [sign up for our newsletter](https://landing.deepset.ai/haystack-community-updates) or [join Haystack discord community](https://discord.gg/haystack).

Thanks for reading!